# 020. Wczytywanie i Eksport Danych (I/O)

Obsługa formatów biznesowych: CSV z polskim formatowaniem liczb, wieloarkuszowy Excel (`ExcelWriter`), zagnieżdżony JSON (`json_normalize`, `explode`) oraz nowoczesny format Parquet.


## 1. CSV z polskimi separatorami dziesiętnymi i kodowaniem


In [ ]:
import pandas as pd
import io


In [ ]:
# Symulacja surowego polskiego pliku CSV
dane_csv_pl = '''id_klienta;nazwisko;kwota_zakupu;data_rejestracji
101;Kowalski, Jan;1 250,50;2024-01-15
102;Nowak, Anna;3 400,00;2024-02-20
103;Wiśniewski, Piotr;890,25;2024-03-05
'''

# Wczytanie z jawnym separatorem kolumn, przecinkiem dziesiętnym i tysięcznym
df_csv = pd.read_csv(
    io.StringIO(dane_csv_pl),
    sep=';',
    decimal=',',
    thousands=' ',
    parse_dates=['data_rejestracji']
)
print("Sparsowana tabela CSV:")
print(df_csv)
print("\nTypy kolumn:\n", df_csv.dtypes)


## 2. Zagnieżdżony JSON z API: `pd.json_normalize()` i `df.explode()`


In [ ]:
dane_api = [
    {
        'zamowienie_id': 1001,
        'klient': {'imie': 'Jan', 'miasto': 'Gdańsk'},
        'pozycje': [
            {'produkt': 'Kawa', 'cena': 25.0},
            {'produkt': 'Ciastko', 'cena': 12.0}
        ]
    },
    {
        'zamowienie_id': 1002,
        'klient': {'imie': 'Anna', 'miasto': 'Kraków'},
        'pozycje': [
            {'produkt': 'Herbata', 'cena': 18.0}
        ]
    }
]


In [ ]:
# Spłaszczanie zagnieżdżonych słowników (klient.imie, klient.miasto)
df_zam = pd.json_normalize(dane_api)
print("Po wykonaniu json_normalize:")
print(df_zam)

In [ ]:
# Rozbicie listy pozycji na osobne wiersze tabeli (explode)
df_rozbite = df_zam.explode('pozycje').reset_index(drop=True)
print("\nPo wykonaniu explode('pozycje'):")
print(df_rozbite[['zamowienie_id', 'klient.imie', 'pozycje']])

## 3. Zapis i odczyt z formatu Parquet (Z kompresją Snappy)


In [ ]:
# Zapis tabeli do wydajnego pliku Parquet
df_csv.to_parquet('klienci.parquet', compression='snappy')

In [ ]:
# Odczyt z filtrowaniem wybranych kolumn (Predicate Pushdown)
df_odczyt = pd.read_parquet('klienci.parquet', columns=['id_klienta', 'kwota_zakupu'])
print("Odczytane z pliku Parquet:")
print(df_odczyt)